# Навчальний ноутбук: від Python-коду до вебзастосунку на Render

## Крок 1. Постановка задачі та короткий опис

**Мета:** створити мінімальний калькулятор на Python, додати вебінтерфейс (Gradio) і підготувати код так, щоб його можна було задеплоїти на Render і отримати робочий URL.

**Функціональні вимоги:**
- Два числові поля введення: `a` та `b`.
- Одна кнопка **«Обчислити»**.
- Два результати: сума `a + b` та різниця `a - b`.
- Підтримка цілих, від'ємних і дробових чисел.
- Порожні або некоректні значення не повинні спричиняти падіння застосунку — показуємо зрозуміле повідомлення про помилку.
- Усі обчислення виконуються Python-кодом на сервері (не в браузері).

**Результат:** користувач відкриває URL у браузері та виконує обчислення без встановлення Python на своєму пристрої.

## Крок 2. Розробка Python-коду

Щоб застосунок був повним і готовим до деплою, потрібно розробити такі елементи:

1. **Обробка помилок** — функція, яка перевіряє і перетворює введені дані (порожній рядок, текст замість числа тощо).
2. **Функція суми** — окрема чиста функція `add(a, b)`.
3. **Функція різниці** — окрема чиста функція `subtract(a, b)`.
4. **Загальна функція `calculate`** — об'єднує перевірку даних і виклик `add`/`subtract`, повертає результат або повідомлення про помилку.
5. **Інтерфейс (Gradio)** — поля вводу, кнопка, поля виводу, прив'язка до `calculate`.
6. **Точка запуску застосунку** (`if __name__ == "__main__":`) — потрібна, щоб Render міг запустити файл командою `python app.py`.

Розглянемо кожен елемент окремо.

### 2.1. Обробка помилок

Функція `parse_number` перетворює текст у число. Якщо значення порожнє або некоректне — повертає `None`, і далі ми покажемо користувачу зрозуміле повідомлення замість падіння застосунку.

In [ ]:
def parse_number(value):
    """Перетворює введений текст у float. Порожнє або некоректне значення -> None."""
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    try:
        return float(text)
    except ValueError:
        return None


# Швидка перевірка
print(parse_number("8"))      # 8.0
print(parse_number("-2.5"))   # -2.5
print(parse_number(""))       # None
print(parse_number("abc"))    # None

### 2.2. Функція суми

In [ ]:
def add(a, b):
    return a + b


print(add(8, 3))     # 11
print(add(-2, 5))    # 3

### 2.3. Функція різниці

In [ ]:
def subtract(a, b):
    return a - b


print(subtract(8, 3))    # 5
print(subtract(-2, 5))   # -7

### 2.4. Загальна функція `calculate`

Об'єднує валідацію вхідних даних і виклик `add`/`subtract`. Саме цю функцію ми підключимо до інтерфейсу.

In [ ]:
def calculate(a, b):
    a_num = parse_number(a)
    b_num = parse_number(b)

    if a_num is None or b_num is None:
        return "Помилка: введіть коректні числа у поля a та b.", ""

    sum_result = add(a_num, b_num)
    diff_result = subtract(a_num, b_num)
    return str(sum_result), str(diff_result)


# Перевірка на прикладах із задачі
print(calculate("8", "3"))       # ('11.0', '5.0')
print(calculate("-2", "5"))      # ('3.0', '-7.0')
print(calculate("2.5", "0.5"))   # ('3.0', '2.0')
print(calculate("0", "0"))       # ('0.0', '0.0')
print(calculate("", "5"))        # повідомлення про помилку

### 2.5. Інтерфейс (Gradio)

Мінімальний вебінтерфейс: два текстових поля, кнопка, два поля результату. Використовуємо `gr.Blocks`, щоб гнучко розмістити елементи рядками.

In [ ]:
import gradio as gr

with gr.Blocks(title="Калькулятор") as demo:
    gr.Markdown("# Мінімальний калькулятор")
    with gr.Row():
        a_input = gr.Textbox(label="a")
        b_input = gr.Textbox(label="b")

    calculate_button = gr.Button("Обчислити")

    with gr.Row():
        sum_output = gr.Textbox(label="Сума (a + b)")
        diff_output = gr.Textbox(label="Різниця (a - b)")

    calculate_button.click(
        fn=calculate,
        inputs=[a_input, b_input],
        outputs=[sum_output, diff_output],
    )

# У ноутбуці можна перевірити інтерфейс так (не обов'язково для деплою):
# demo.launch()

### 2.6. Точка запуску застосунку

Щоб Render міг запустити застосунок командою `python app.py`, потрібен блок `if __name__ == "__main__":`, який:
- бере порт із змінної середовища `PORT` (Render сам її надає; локально використовуємо значення за замовчуванням);
- запускає сервер на адресі `0.0.0.0` (а не `127.0.0.1`), інакше Render не зможе проксіювати трафік ззовні.

In [ ]:
import os

# У файлі app.py цей блок запускає сервер, коли файл виконують напряму (python app.py).
# У цьому ноутбуці ми НЕ викликаємо demo.launch() всередині if-блоку,
# бо це заблокує виконання комірок ноутбука (сервер працює, доки його не зупинити).
# Щоб перевірити інтерфейс наживо — збережіть код у app.py і запустіть `python app.py` у терміналі.
if __name__ == "__main__":
    port = int(os.environ.get("PORT", 7860))
    print(f"У app.py тут буде: demo.launch(server_name='0.0.0.0', server_port={port})")

### 2.7. Що ще потрібно, щоб застосунок «злетів» на Render

Крім самого коду `app.py`, для успішного деплою потрібні ще кілька елементів проєкту:

| Файл / елемент | Навіщо |
| --- | --- |
| `requirements.txt` | Render виконує `pip install -r requirements.txt` — без файлу залежності не встановляться. |
| Запуск на `0.0.0.0` + порт з `PORT` | Без цього Render не зможе достукатися до застосунку ззовні (вже реалізовано вище). |
| `if __name__ == "__main__":` | Без цього блоку `python app.py` нічого не запустить. |
| `.gitignore` (`.venv/`, `__pycache__/`, `.env`) | Щоб віртуальне середовище й секрети не потрапили в GitHub і не зламали збірку на Render. |
| Зафіксовані версії в `requirements.txt` (напр. `gradio==5.49.1`) | Гарантує, що на сервері встановиться та сама версія, що й локально — без несподіваних несумісностей. |
| `README.md` | Інструкція для інших людей (і для ментора) — як запустити локально та де знайти опублікований URL. |
| Узгоджена версія Python | Render використовує Python за замовчуванням або вказаний у налаштуваннях сервісу — варто перевірити, що локальна версія (напр. 3.13) підтримується; за потреби додати файл `runtime.txt` з версією. |
| Build/Start commands у Render | Build: `pip install -r requirements.txt`; Start: `python app.py` — налаштовуються в самому Render, не у коді. |

**Чого НЕ потрібно для цієї задачі:** Dockerfile, FastAPI, база даних, `share=True` у `demo.launch()` (тимчасове посилання Gradio не потрібне — Render вже дає постійний HTTPS URL).

### 2.8. `requirements.txt`

Файл із залежностями проєкту. Версії краще фіксувати (`==`), щоб на сервері встановилось те саме, що перевірялось локально.

In [ ]:
requirements_txt = """gradio==5.49.1
audioop-lts; python_version >= \"3.13\"
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_txt)

print(requirements_txt)

## Крок 3. Git / GitHub — основні поняття та команди

**Основні поняття:**
- **Git** — система контролю версій: зберігає історію змін коду локально.
- **Репозиторій (repo)** — папка проєкту під контролем Git.
- **Commit** — «знімок» змін із описом, що саме змінилось.
- **Branch (гілка)** — незалежна лінія розробки (напр. `main`, `improve-ui`).
- **Remote** — віддалений репозиторій (GitHub), з яким синхронізується локальний.
- **Push / Pull** — відправка змін на GitHub / отримання змін із GitHub.
- **Pull Request (PR)** — запит на злиття змін із однієї гілки в іншу, зручний для перевірки перед `merge`.

**Корисні команди:**

```bash
git init                          # ініціалізувати репозиторій
git status                        # що змінилось / що не відстежується
git add .                         # додати всі зміни до staging
git commit -m "Опис змін"         # зафіксувати зміни
git branch improve-ui             # створити нову гілку
git checkout improve-ui           # переключитись на гілку
git checkout -b improve-ui        # створити і одразу переключитись

git remote add origin <URL>       # підключити GitHub-репозиторій
git push -u origin main           # відправити гілку main вперше
git push                          # відправити зміни далі

git pull                          # отримати оновлення з GitHub
git log --oneline                 # коротка історія комітів
```

**Типовий робочий цикл для цієї задачі:**
1. `git init` → `git status` → `git add .` → `git commit -m "Initial calculator app"`.
2. Створити репозиторій на GitHub, підключити як `origin`, зробити `git push -u origin main`.
3. Для змін: `git checkout -b improve-ui` → редагувати код → `git add .` → `git commit` → `git push -u origin improve-ui` → відкрити Pull Request на GitHub → `merge` у `main`.
4. Переконатись, що `.venv/` і секрети не потрапили в репозиторій (перевіряє `.gitignore`).

### 3.1. Як відправити код у репозиторій — покроково

Заміни `HereWillBeYouRepoURL` на URL свого репозиторію на GitHub (його видно на сторінці репозиторію під кнопкою **Code**).

```bash
# 1. Перейти в папку проєкту
cd шлях/до/проєкту

# 2. Ініціалізувати git (лише один раз для нового проєкту)
git init

# 3. Перевірити, які файли будуть закомічені
git status

# 4. Додати файли до staging
git add .

# 5. Зробити перший commit
git commit -m "Initial calculator app"

# 6. Перейменувати гілку на main (якщо вона ще не так називається)
git branch -M main

# 7. Підключити GitHub-репозиторій як remote
git remote add origin HereWillBeYouRepoURL

# 8. Відправити код на GitHub
git push -u origin main
```

**Перевірка:** відкрий `HereWillBeYouRepoURL` у браузері — файли `app.py`, `requirements.txt`, `.gitignore`, `README.md` мають з'явитись у репозиторії.

**Якщо щось змінив пізніше:**

```bash
git add .
git commit -m "Опис змін"
git push
```

**Типові проблеми:**
- `Authentication failed` / пароль не підходить — GitHub більше не приймає звичайний пароль для push по HTTPS, потрібен **Personal Access Token** замість пароля, або налаштований SSH-ключ.
- `remote origin already exists` — remote вже доданий раніше; подивись поточний командою `git remote -v`, за потреби онови: `git remote set-url origin HereWillBeYouRepoURL`.
- У репозиторій потрапив `.venv` або інші зайві файли — перевір, що вони перелічені в `.gitignore`, і виконай `git rm -r --cached .venv` перед новим commit.

## Крок 4. Деплой на Render

Render — хостинг, який запускає Python-застосунок за кодом із GitHub і дає постійний HTTPS URL. Ноутбук тут не виконує код — це опис дій у вебінтерфейсі Render.

### 4.1. Створення Web Service — покроково

1. Зареєструватись / увійти на [render.com](https://render.com).
2. **New +** → **Web Service**.
3. Підключити GitHub і обрати репозиторій з кодом калькулятора (`HereWillBeYouRepoURL`).
4. Обрати:
   - **Runtime:** Python 3
   - **Instance Type:** Free (якщо доступний)
5. Задати команди:
   - **Build Command:** `pip install -r requirements.txt`
   - **Start Command:** `python app.py`
6. Натиснути **Create Web Service** — Render виконає перший build і deploy, за цим можна стежити в вкладці **Logs**.
7. Після успішного деплою Render видасть URL виду `https://<назва-сервісу>.onrender.com` — це і є опублікований застосунок.

### 4.2. Навіщо потрібен `PORT` і `0.0.0.0`

Render сам призначає порт і передає його застосунку через змінну середовища `PORT`. Якщо застосунок слухає фіксований порт або `127.0.0.1`, Render не зможе достукатись до нього ззовні. Саме тому в `app.py` є:

```python
if __name__ == "__main__":
    port = int(os.environ.get("PORT", 7860))
    demo.launch(server_name="0.0.0.0", server_port=port)
```

- `os.environ.get("PORT", 7860)` — бере порт від Render, а якщо його немає (локальний запуск) — використовує `7860` за замовчуванням.
- `server_name="0.0.0.0"` — слухає всі мережеві інтерфейси, а не лише локальну машину.

### 4.3. Реальна проблема, яку ми зустріли, і як її вирішили

**Симптом:** білд падав з помилкою на кроці встановлення залежностей:

```
Preparing metadata (pyproject.toml): finished with status 'error'
...
error: failed to create directory `/usr/local/cargo/registry/cache/...`
Caused by:
  Read-only file system (os error 30)
💥 maturin failed
```

**Причина:** Render за замовчуванням використав найновіший **Python 3.14**, для якого ще немає готового (prebuilt) wheel-файлу пакета `pydantic-core` (залежність `gradio` → `fastapi` → `pydantic`). Pip намагався зібрати пакет із Rust-коду через `maturin`, а файлова система білд-середовища — read-only для кешу `cargo`, тому збірка падала.

**Рішення:**
1. Додати файл [`runtime.txt`](runtime.txt) у корінь репозиторію з вмістом `python-3.12.7` — як документація/запасний варіант.
2. **Головне:** у Render Dashboard → вкладка **Environment** → додати змінну середовища:
   - **Key:** `PYTHON_VERSION`
   - **Value:** `3.12.7` (саме повна версія, не `3.12`)
3. Зберегти та запустити **Manual Deploy**.

**Чому так:** Render пріоритетно читає версію Python зі змінної середовища `PYTHON_VERSION` у Dashboard, а `runtime.txt` у репозиторії — лише запасний орієнтир, який у нашому випадку був проігнорований. Python 3.12 має готові wheels для всіх залежностей `gradio`, тому збірка з Rust/maturin більше не потрібна — build проходить швидко й без помилок.

**Висновок для майбутніх проєктів:** якщо build на Render падає на етапі встановлення залежностей з незрозумілою помилкою компіляції (Rust/C-розширення, read-only filesystem тощо) — варто перевірити версію Python і спробувати зафіксувати стабільнішу/трохи старішу версію через `PYTHON_VERSION` у Dashboard.

### 4.4. Перевірка опублікованого застосунку

- Відкрити URL виду `https://<назва-сервісу>.onrender.com` у браузері.
- Прогнати приклади з таблиці задачі (`8,3 → 11,5`; `-2,5 → 3,-7`; `2.5,0.5 → 3,2`; `0,0 → 0,0`).
- Перевірити порожнє поле — має з'явитись повідомлення про помилку, а не збій сторінки.
- Зупинити локальний застосунок (`Ctrl+C` у терміналі) і переконатись, що URL на Render працює незалежно від локального комп'ютера.
- Відкрити URL у двох вкладках/пристроях одночасно — переконатись, що введені дані не змішуються.
- Якщо сервіс на безкоштовному плані "заснув" (не було запитів деякий час) — перший запит може займати 30-60 секунд на "пробудження".